In [1]:
# Cell 1: Authenticate Earth Engine, import dependencies, and prepare the project data folder.
import hashlib
import json
import time
from math import ceil, floor
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import numpy as np
import rasterio

from affine import Affine
from rasterio.windows import Window
from shapely.geometry import Point

ee.Authenticate()
ee.Initialize()

cwd = Path.cwd()
project_dir = cwd.parent if cwd.name.lower() in {"notebook", "notebooks"} else cwd
data_dir = project_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)
print(f"Input data folder: {data_dir.resolve()}")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Input data folder: D:\HydropowerScreening\data


In [2]:
# Cell 2: Explain how to draw a buffered rectangular AOI and outlet point.
from IPython.display import Markdown, display

display(Markdown("""
## Before you draw

Draw **one rectangle** that fully contains the catchment and leaves a generous buffer on every side, then draw **one outlet point** inside that rectangle. The buffer is important: the final DEM is trimmed to remove projected edge NoData, and a larger extent also prevents the catchment from being clipped during delineation. Draw both shapes on the map below before running the next cell. Large areas are downloaded into a temporary tiles folder under `data/`, mosaicked into one DEM, and cleaned up automatically.
"""))


## Before you draw

Draw **one rectangle** that fully contains the catchment and leaves a generous buffer on every side, then draw **one outlet point** inside that rectangle. The buffer is important: the final DEM is trimmed to remove projected edge NoData, and a larger extent also prevents the catchment from being clipped during delineation. Draw both shapes on the map below before running the next cell. Large areas are downloaded into a temporary tiles folder under `data/`, mosaicked into one DEM, and cleaned up automatically.


In [3]:
# Cell 3: Load GLO-30 and open the interactive map for drawing the AOI and outlet.
Map = geemap.Map(center=[20, 0], zoom=2)
glo = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM")
proj = glo.first().projection()
dem = glo.mosaic().setDefaultProjection(proj)
Map.addLayer(
    dem,
    {"min": 0, "max": 8000, "palette": ["#173b8f", "#239bd1", "#67bd63", "#f6e65c", "#e88232", "#f4f1e8"]},
    "GLO-30 elevation",
)
Map

Map(center=[20, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', trans…

In [5]:
# Cell 4: Validate and save the user-drawn watershed rectangle and outlet point.
drawn = Map.user_rois
if drawn is None:
    raise ValueError("Draw one rectangle and one outlet point on the map, then rerun this cell.")

features = drawn.getInfo().get("features", [])
polygon_features = [f for f in features if f.get("geometry", {}).get("type") in ("Polygon", "MultiPolygon")]
point_features = [f for f in features if f.get("geometry", {}).get("type") == "Point"]
if len(polygon_features) != 1 or len(point_features) != 1:
    raise ValueError(
        f"Expected exactly one rectangle and one point; found {len(polygon_features)} polygon(s) and {len(point_features)} point(s)."
    )

aoi_geometry = polygon_features[0]["geometry"]
outlet_geometry = point_features[0]["geometry"]
aoi = ee.Geometry(aoi_geometry)

with open(data_dir / "watershedAOI.geojson", "w", encoding="utf-8") as f:
    json.dump(
        {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {"name": "watershed"}, "geometry": aoi_geometry}]},
        f,
        indent=2,
    )

outlet_coords = outlet_geometry["coordinates"]
outlet_gdf = gpd.GeoDataFrame(
    {"name": ["outlet"]},
    geometry=[Point(outlet_coords[0], outlet_coords[1])],
    crs="EPSG:4326",
)
outlet_path = data_dir / "outlet.shp"
outlet_gdf.to_file(outlet_path)
print(f"AOI saved: {data_dir / 'watershedAOI.geojson'}")
print(f"Outlet saved: {outlet_path}")

AOI saved: d:\HydropowerScreening\data\watershedAOI.geojson
Outlet saved: d:\HydropowerScreening\data\outlet.shp


In [6]:
# Cell 5: Download the AOI DEM in resumable tiles, then mosaic them into one raw DEM.
source_dem_path = data_dir / "sourceDEM.tif"
tile_bounds = aoi.bounds(maxError=1).coordinates().getInfo()[0]
tile_key = hashlib.sha256(json.dumps(tile_bounds, sort_keys=True).encode("utf-8")).hexdigest()[:12]
tiles_dir = data_dir / f"dem_tiles_{tile_key}"
tiles_dir.mkdir(parents=True, exist_ok=True)


def is_valid_tile(path):
    if not path.exists():
        return False
    try:
        with rasterio.open(path) as src:
            valid = src.width > 0 and src.height > 0 and src.count == 1
        if not valid:
            path.unlink()
        return valid
    except rasterio.errors.RasterioError:
        path.unlink()
        return False

if source_dem_path.exists() and not is_valid_tile(source_dem_path):
    print("Removed an invalid partial source DEM from an interrupted run.")


def export_tile(region, path, label, depth=0, max_depth=6):
    if is_valid_tile(path):
        print(f"Reusing downloaded {label}: {path.name}")
        return [path]

    try:
        geemap.ee_export_image(
            dem.clip(region),
            filename=str(path),
            scale=30,
            region=region,
            file_per_band=False,
        )
        if not is_valid_tile(path):
            raise ValueError(f"The {label} download is empty or invalid.")
        print(f"Downloaded {label}: {path.name}")
        time.sleep(0.5)
        return [path]
    except Exception as error:
        if path.exists():
            path.unlink()
        if depth >= max_depth:
            raise RuntimeError(f"Could not download {label} after {max_depth} subdivisions.") from error
        print(f"{label} download failed; subdividing it (level {depth + 1}/{max_depth}): {error}")
        bounds = region.bounds(maxError=1).coordinates().getInfo()[0]
        xmin = min(point[0] for point in bounds)
        xmax = max(point[0] for point in bounds)
        ymin = min(point[1] for point in bounds)
        ymax = max(point[1] for point in bounds)
        if xmin == xmax or ymin == ymax:
            raise RuntimeError(f"Cannot subdivide the zero-area region for {label}.") from error
        xmid = (xmin + xmax) / 2
        ymid = (ymin + ymax) / 2
        quadrants = [
            ("NW", ee.Geometry.Rectangle([xmin, ymid, xmid, ymax])),
            ("NE", ee.Geometry.Rectangle([xmid, ymid, xmax, ymax])),
            ("SW", ee.Geometry.Rectangle([xmin, ymin, xmid, ymid])),
            ("SE", ee.Geometry.Rectangle([xmid, ymin, xmax, ymid])),
        ]
        paths = []
        for suffix, subregion in quadrants:
            subpath = tiles_dir / f"{path.stem}_{suffix}{path.suffix}"
            paths.extend(export_tile(subregion, subpath, f"{label}-{suffix}", depth + 1, max_depth))
        return paths


try:
    print("Attempting one DEM download for the full rectangle...")
    downloaded_tiles = export_tile(aoi, tiles_dir / "sourceDEM.tif", "watershed AOI")
except RuntimeError:
    raise
except Exception as error:
    raise RuntimeError("DEM download failed; check Earth Engine access and the selected rectangle.") from error

partial_dem_path = data_dir / "sourceDEM.partial.tif"
if partial_dem_path.exists():
    partial_dem_path.unlink()
print(f"Mosaicking {len(downloaded_tiles)} DEM tiles in 512 x 512 pixel windows...")
with rasterio.open(downloaded_tiles[0]) as reference:
    resolution_x, _ = reference.res
    tile_bounds = []
    for tile_path in downloaded_tiles:
        with rasterio.open(tile_path) as tile:
            tile_bounds.append(tile.bounds)
    left = min(bounds.left for bounds in tile_bounds)
    bottom = min(bounds.bottom for bounds in tile_bounds)
    right = max(bounds.right for bounds in tile_bounds)
    top = max(bounds.top for bounds in tile_bounds)
    col_start = floor((left - reference.transform.c) / resolution_x)
    col_stop = ceil((right - reference.transform.c) / resolution_x)
    row_start = floor((top - reference.transform.f) / reference.transform.e)
    row_stop = ceil((bottom - reference.transform.f) / reference.transform.e)
    output_transform = reference.transform * Affine.translation(col_start, row_start)
    output_width = col_stop - col_start
    output_height = row_stop - row_start
    if output_width <= 0 or output_height <= 0:
        raise ValueError("The DEM tile mosaic has invalid dimensions.")
    output_profile = reference.profile.copy()
    output_profile.update(
        driver="GTiff",
        width=output_width,
        height=output_height,
        transform=output_transform,
        tiled=True,
        blockxsize=512,
        blockysize=512,
        compress="deflate",
    )
    fill_value = reference.nodata if reference.nodata is not None else 0
    dtype = np.dtype(reference.dtypes[0])

    with rasterio.open(partial_dem_path, "w", **output_profile) as destination:
        for row in range(0, output_height, 512):
            height = min(512, output_height - row)
            for col in range(0, output_width, 512):
                width = min(512, output_width - col)
                window = Window(col, row, width, height)
                destination.write(np.full((height, width), fill_value, dtype=dtype), 1, window=window)

        for index, tile_path in enumerate(downloaded_tiles, start=1):
            with rasterio.open(tile_path) as tile:
                if tile.crs != reference.crs or tile.count != 1:
                    raise ValueError(f"Tile {tile_path.name} has a different CRS or band count.")
                if abs(tile.transform.a - reference.transform.a) > 1e-12 or abs(tile.transform.e - reference.transform.e) > 1e-12:
                    raise ValueError(f"Tile {tile_path.name} has a different pixel resolution.")
                col = round((tile.transform.c - output_transform.c) / resolution_x)
                row = round((tile.transform.f - output_transform.f) / reference.transform.e)
                if abs((tile.transform.c - output_transform.c) / resolution_x - col) > 1e-4 or abs((tile.transform.f - output_transform.f) / reference.transform.e - row) > 1e-4:
                    raise ValueError(f"Tile {tile_path.name} is not aligned to the output pixel grid.")
                if col < 0 or row < 0 or col + tile.width > output_width or row + tile.height > output_height:
                    raise ValueError(f"Tile {tile_path.name} falls outside the output mosaic bounds.")
                destination.write(
                    tile.read(1),
                    1,
                    window=Window(col, row, tile.width, tile.height),
                )
            print(f"Mosaicked tile {index}/{len(downloaded_tiles)}: {tile_path.name}")

if not is_valid_tile(partial_dem_path):
    raise RuntimeError("The merged source DEM is empty or invalid; downloaded tiles were retained for retry.")
partial_dem_path.replace(source_dem_path)

with rasterio.open(source_dem_path) as src:
    print(f"Source DEM: {source_dem_path}")
    print("CRS        :", src.crs)
    print("Dimensions :", src.width, "x", src.height)
    print("Resolution:", src.res)
    print("Bounds     :", src.bounds)
    print("NoData     :", src.nodata)

for tile_path in tiles_dir.iterdir():
    if tile_path.is_file():
        tile_path.unlink()
tiles_dir.rmdir()
print(f"Merged {len(downloaded_tiles)} tiles; temporary tiles removed.")

Attempting one DEM download for the full rectangle...
Generating URL ...
An error occurred while downloading.
Total request size (271341000 bytes) must be less than or equal to 50331648 bytes.
watershed AOI download failed; subdividing it (level 1/6): The watershed AOI download is empty or invalid.
Generating URL ...
An error occurred while downloading.
Total request size (67908600 bytes) must be less than or equal to 50331648 bytes.
watershed AOI-NW download failed; subdividing it (level 2/6): The watershed AOI-NW download is empty or invalid.
Generating URL ...
Please wait ...
Data downloaded to d:\HydropowerScreening\data\dem_tiles_3894030f446a\sourceDEM_NW_NW.tif
Downloaded watershed AOI-NW-NW: sourceDEM_NW_NW.tif
Generating URL ...
Please wait ...
Data downloaded to d:\HydropowerScreening\data\dem_tiles_3894030f446a\sourceDEM_NW_NE.tif
Downloaded watershed AOI-NW-NE: sourceDEM_NW_NE.tif
Generating URL ...
Please wait ...
Data downloaded to d:\HydropowerScreening\data\dem_tiles_389